<a href="https://colab.research.google.com/github/sachin23-an/Quant-Finance-Projects/blob/main/Market_Timing_Optimization_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Market Timing Optimization Model for NIFTY 50

This project develops and backtests a quantitative market timing strategy for the NIFTY 50 index. The core idea is to identify optimal entry points based on a combination of momentum, volatility, and market drawdown indicators. We aim to demonstrate how a rule-based system can potentially improve risk-adjusted returns compared to a simple Buy & Hold strategy.

**Key Steps Include:**

1.  **Data Acquisition:** Downloading historical daily price data for the NIFTY 50 index.
2.  **Feature Engineering:** Calculating technical indicators such as moving averages, momentum strength, rolling volatility, and market drawdown.
3.  **Entry Score Development:** Creating a composite 'Entry Score' by normalizing and combining these features to quantify market attractiveness.
4.  **Position Sizing Strategy:** Defining capital allocation rules based on the calculated Entry Score.
5.  **Backtesting:** Simulating the strategy's performance against a Buy & Hold benchmark using historical data.
6.  **Visualization:** Plotting portfolio growth, entry scores, drawdowns, and identified 'buy zones'.
7.  **Performance Summary:** Calculating key metrics like Total Return, CAGR, Max Drawdown, and Sharpe Ratio for both strategies.
8.  **Interactive Adjustment:** Providing an interactive tool to explore how different weightings of the Entry Score components impact the strategy's performance.

**Objective:** To design and evaluate a transparent, rule-based market timing model, providing insights into its potential benefits and limitations in managing exposure to the NIFTY 50 index.

In [1]:
# --- Initial Setup and Library Imports ---
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import gmean
from IPython.display import display, HTML
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, VBox, HBox

# Set display options for pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("Libraries imported successfully.")

Libraries imported successfully.


## STEP 1: Data Acquisition

Downloading 8 years of daily data for ^NSEI (NIFTY 50) using `yfinance`.

In [2]:
# Define the ticker and date range
TICKER = '^NSEI'  # NIFTY 50
END_DATE = pd.Timestamp.now().strftime('%Y-%m-%d')
START_DATE = (pd.Timestamp.now() - pd.DateOffset(years=8)).strftime('%Y-%m-%d')

print(f"Downloading {TICKER} data from {START_DATE} to {END_DATE}")

# Download data
df_raw = yf.download(TICKER, start=START_DATE, end=END_DATE)

# Initialize df
df = pd.DataFrame()

# Handle MultiIndex or single-level columns for 'Close' or 'Adj Close'
# Check if the columns are a MultiIndex
if isinstance(df_raw.columns, pd.MultiIndex):
    close_col_candidates = [col for col in df_raw.columns if 'Close' in col or 'Adj Close' in col]
    if len(close_col_candidates) > 0:
        # Prioritize 'Close' over 'Adj Close' if both exist, and pick the first one found
        preferred_col = None
        for col in close_col_candidates:
            if 'Close' in col:
                preferred_col = col
                break
        if preferred_col is None and len(close_col_candidates) > 0:
            preferred_col = close_col_candidates[0] # Fallback to first available if 'Close' not found explicitly

        df = df_raw[preferred_col].to_frame()
        df.columns = ['Close'] # Ensure single column name
    else:
        raise ValueError("Cannot identify 'Close' or 'Adj Close' in MultiIndex columns.")
elif 'Close' in df_raw.columns:
    df = df_raw[['Close']]
elif 'Adj Close' in df_raw.columns:
    df = df_raw[['Adj Close']].rename(columns={'Adj Close': 'Close'})
else:
    raise ValueError("Neither 'Close' nor 'Adj Close' column found in downloaded data.")

# Ensure the final DataFrame has a single 'Close' column and a simple index
df.columns = ['Close']
df.index.name = 'Date'

print(f"Data downloaded successfully. Shape: {df.shape}")
display(df.head())
display(df.tail())

/tmp/ipykernel_18671/2392628270.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_raw = yf.download(TICKER, start=START_DATE, end=END_DATE)
[*********************100%***********************]  1 of 1 completed

Data downloaded successfully. Shape: (1969, 1)


,Close
Date,
2018-04-23,10584.700195
2018-04-24,10614.349609
2018-04-25,10570.549805
2018-04-26,10617.799805
2018-04-27,10692.299805


,Close
Date,
2026-04-13,23842.650391
2026-04-15,24231.300781
2026-04-16,24196.750000
2026-04-17,24353.550781
2026-04-20,24364.849609


In [3]:
# Calculate Moving Averages
df['MA50'] = df['Close'].rolling(window=50).mean()
df['MA200'] = df['Close'].rolling(window=200).mean()

# Calculate Momentum Strength
df['MomentumStrength'] = (df['Close'] / df['MA200']) - 1

# Calculate Daily Returns
df['DailyReturns'] = df['Close'].pct_change()

# Calculate 20-day Rolling Volatility (standard deviation of daily returns)
# This will be normalized later
df['Volatility'] = df['DailyReturns'].rolling(window=20).std()

# Calculate Rolling Maximum Price
df['RollingMax'] = df['Close'].expanding().max() # Or use .cummax()

# Calculate Drawdown
df['Drawdown'] = (df['Close'] - df['RollingMax']) / df['RollingMax']

# Drop NaN values that resulted from rolling calculations
df.dropna(inplace=True)

print("Features calculated successfully.")
display(df.head())

Features calculated successfully.


,Close,MA50,MA200,MomentumStrength,DailyReturns,Volatility,RollingMax,Drawdown
Date,,,,,,,,
2019-02-11,10888.799805,10817.152988,10855.679238,0.003051,-0.005007,0.007518,11738.5,-0.072386
2019-02-12,10831.400391,10816.245996,10856.912739,-0.002350,-0.005271,0.006951,11738.5,-0.077276
2019-02-14,10746.049805,10813.491992,10857.571240,-0.010271,-0.007880,0.007156,11738.5,-0.084547
2019-02-15,10724.400391,10810.590000,10858.340493,-0.012335,-0.002015,0.007146,11738.5,-0.086391
2019-02-18,10640.950195,10807.750996,10858.456245,-0.020031,-0.007781,0.007308,11738.5,-0.093500


## STEP 2: Feature Engineering

Calculating the required features:
- 50-day moving average (MA50)
- 200-day moving average (MA200)
- Momentum Strength = (Price / MA200) - 1
- 20-day rolling volatility (std dev of daily returns), normalized
- Rolling maximum price
- Drawdown = (Price - Rolling Max) / Rolling Max

## STEP 3: Entry Score

Creating a composite Entry Score by combining:
- Momentum component (positive momentum = higher score)
- Volatility component (lower volatility = higher score)
- Drawdown component (deeper drawdown = buying opportunity)

Normalizing all 3 components to 0-1 range. Combining them equally by default, but ready for interactive weighting.

In [4]:
# --- Entry Score Calculation ---

# Normalize MomentumStrength to 0-1 range (higher is better)
# Since MomentumStrength can be negative, we shift it to be non-negative
# before scaling. A simple approach is min-max scaling after ensuring a positive range.
min_momentum = df['MomentumStrength'].min()
max_momentum = df['MomentumStrength'].max()

# Handle case where min_momentum == max_momentum to avoid division by zero
if (max_momentum - min_momentum) == 0:
    df['MomentumScore'] = 0.5 # Assign a neutral score if no variation
else:
    df['MomentumScore'] = (df['MomentumStrength'] - min_momentum) / (max_momentum - min_momentum)

# Normalize Volatility to 0-1 range (lower is better, so invert after normalization)
min_volatility = df['Volatility'].min()
max_volatility = df['Volatility'].max()

# Handle case where min_volatility == max_volatility
if (max_volatility - min_volatility) == 0:
    df['VolatilityScore'] = 0.5
else:
    # Invert the score: 1 - normalized_volatility so lower volatility gets a higher score
    df['VolatilityScore'] = 1 - ((df['Volatility'] - min_volatility) / (max_volatility - min_volatility))


# Normalize Drawdown to 0-1 range (deeper drawdown is better, so invert after normalization)
# Drawdown is typically negative or zero. A deeper drawdown means a more negative value.
# To get a higher score for deeper drawdowns, we normalize (most negative -> 1, 0 -> 0)
min_drawdown = df['Drawdown'].min()
max_drawdown = df['Drawdown'].max() # This will typically be 0 or close to 0

# Handle case where min_drawdown == max_drawdown
if (max_drawdown - min_drawdown) == 0:
    df['DrawdownScore'] = 0.5
else:
    # Normalize so that min_drawdown (deepest) maps to 1, and max_drawdown (shallowest/0) maps to 0.
    df['DrawdownScore'] = 1 - (df['Drawdown'] - min_drawdown) / (max_drawdown - min_drawdown)

# Combine scores with equal weighting for now (will be made interactive later)
def calculate_entry_score(df_input, momentum_weight, volatility_weight, drawdown_weight):
    df_output = df_input.copy()
    # Auto-normalize weights if their sum is not 1
    total_weight = momentum_weight + volatility_weight + drawdown_weight
    if total_weight == 0:
        # Default to equal weighting if all weights are zero
        norm_m_w, norm_v_w, norm_d_w = 1/3, 1/3, 1/3
    else:
        norm_m_w = momentum_weight / total_weight
        norm_v_w = volatility_weight / total_weight
        norm_d_w = drawdown_weight / total_weight

    df_output['EntryScore'] = (
        norm_m_w * df_output['MomentumScore'] +
        norm_v_w * df_output['VolatilityScore'] +
        norm_d_w * df_output['DrawdownScore']
    )
    return df_output

# Calculate initial EntryScore with equal weights
df = calculate_entry_score(df, 1, 1, 1)

print("Entry Score and components calculated successfully.")
display(df[['Close', 'MomentumScore', 'VolatilityScore', 'DrawdownScore', 'EntryScore']].head())
display(df[['Close', 'MomentumScore', 'VolatilityScore', 'DrawdownScore', 'EntryScore']].tail())

Entry Score and components calculated successfully.


,Close,MomentumScore,VolatilityScore,DrawdownScore,EntryScore
Date,,,,,
2019-02-11,10888.799805,0.536588,0.917628,0.188309,0.547508
2019-02-12,10831.400391,0.528175,0.928364,0.201030,0.552523
2019-02-14,10746.049805,0.515836,0.924490,0.219945,0.553424
2019-02-15,10724.400391,0.512621,0.924680,0.224743,0.554015
2019-02-18,10640.950195,0.500633,0.921612,0.243237,0.555161


,Close,MomentumScore,VolatilityScore,DrawdownScore,EntryScore
Date,,,,,
2026-04-13,23842.650391,0.449632,0.718547,0.245626,0.471268
2026-04-15,24231.300781,0.473946,0.714809,0.207225,0.465326
2026-04-16,24196.750000,0.472038,0.728662,0.210639,0.470446
2026-04-17,24353.550781,0.481952,0.730407,0.195145,0.469168
2026-04-20,24364.849609,0.482915,0.731183,0.194029,0.469376


In [5]:
# Calculate Moving Averages
df['MA50'] = df['Close'].rolling(window=50).mean()
df['MA200'] = df['Close'].rolling(window=200).mean()

# Calculate Momentum Strength
df['MomentumStrength'] = (df['Close'] / df['MA200']) - 1

# Calculate Daily Returns
df['DailyReturns'] = df['Close'].pct_change()

# Calculate 20-day Rolling Volatility (standard deviation of daily returns)
# This will be normalized later
df['Volatility'] = df['DailyReturns'].rolling(window=20).std()

# Calculate Rolling Maximum Price
df['RollingMax'] = df['Close'].expanding().max() # Or use .cummax()

# Calculate Drawdown
df['Drawdown'] = (df['Close'] - df['RollingMax']) / df['RollingMax']

# Drop NaN values that resulted from rolling calculations
df.dropna(inplace=True)

print("Features calculated successfully.")
display(df.head())

Features calculated successfully.


,Close,MA50,MA200,MomentumStrength,DailyReturns,Volatility,RollingMax,Drawdown,MomentumScore,VolatilityScore,DrawdownScore,EntryScore
Date,,,,,,,,,,,,
2019-12-11,11910.150391,11754.952012,11471.061528,0.038278,0.004500,0.005386,12151.150391,-0.019834,0.591461,0.958008,0.051596,0.533689
2019-12-12,11971.799805,11762.964004,11476.476528,0.043160,0.005176,0.005472,12151.150391,-0.014760,0.599066,0.956374,0.038398,0.531279
2019-12-13,12086.700195,11774.450000,11482.753027,0.052596,0.009598,0.005839,12151.150391,-0.005304,0.613765,0.949435,0.013798,0.525666
2019-12-16,12053.950195,11786.040000,11489.292529,0.049146,-0.002710,0.005880,12151.150391,-0.007999,0.608391,0.948648,0.020810,0.525950
2019-12-17,12165.000000,11802.141992,11496.495527,0.058149,0.009213,0.006123,12165.000000,0.000000,0.622414,0.944048,0.000000,0.522154


## STEP 4: Position Sizing Strategy

Based on the Entry Score, we will define the investment allocation:
- High Score (above 0.6)  --> 100% invested
- Medium Score (0.3-0.6)  --> 50% invested
- Low Score (below 0.3)   --> 15% invested

In [6]:
# --- Position Sizing Strategy ---

def apply_position_sizing(df_input):
    df_output = df_input.copy()
    df_output['Position'] = 0.0

    # High Score (above 0.6) --> 100% invested
    df_output.loc[df_output['EntryScore'] > 0.6, 'Position'] = 1.0

    # Medium Score (0.3-0.6) --> 50% invested
    df_output.loc[(df_output['EntryScore'] >= 0.3) & (df_output['EntryScore'] <= 0.6), 'Position'] = 0.5

    # Low Score (below 0.3) --> 15% invested
    df_output.loc[df_output['EntryScore'] < 0.3, 'Position'] = 0.15

    return df_output

df = apply_position_sizing(df)

print("Position sizing applied successfully.")
display(df[['Close', 'EntryScore', 'Position']].head())
display(df[['Close', 'EntryScore', 'Position']].tail())

Position sizing applied successfully.


,Close,EntryScore,Position
Date,,,
2019-12-11,11910.150391,0.533689,0.5
2019-12-12,11971.799805,0.531279,0.5
2019-12-13,12086.700195,0.525666,0.5
2019-12-16,12053.950195,0.525950,0.5
2019-12-17,12165.000000,0.522154,0.5


,Close,EntryScore,Position
Date,,,
2026-04-13,23842.650391,0.471268,0.5
2026-04-15,24231.300781,0.465326,0.5
2026-04-16,24196.750000,0.470446,0.5
2026-04-17,24353.550781,0.469168,0.5
2026-04-20,24364.849609,0.469376,0.5


## STEP 5: Backtest

Simulating an INR 100,000 starting capital for both the strategy and a Buy & Hold approach, calculating daily portfolio values.

In [7]:
# --- Backtesting ---

INITIAL_CAPITAL = 100000

# Calculate daily returns for the strategy
df['StrategyDailyReturns'] = df['DailyReturns'] * df['Position'].shift(1) # Shift position by 1 day as decisions are based on previous day's close

# Calculate Buy & Hold returns
df['BuyHoldDailyReturns'] = df['DailyReturns']

# Initialize portfolio values
df['StrategyPortfolioValue'] = INITIAL_CAPITAL * (1 + df['StrategyDailyReturns']).cumprod()
df['BuyHoldPortfolioValue'] = INITIAL_CAPITAL * (1 + df['BuyHoldDailyReturns']).cumprod()

# Fill initial NaN values (due to .shift(1) and .cumprod())
# Using assign or direct assignment to avoid FutureWarning
df['StrategyPortfolioValue'] = df['StrategyPortfolioValue'].fillna(INITIAL_CAPITAL)
df['BuyHoldPortfolioValue'] = df['BuyHoldPortfolioValue'].fillna(INITIAL_CAPITAL)

print("Backtest simulation completed.")
display(df[['Close', 'StrategyDailyReturns', 'StrategyPortfolioValue', 'BuyHoldPortfolioValue']].head())
display(df[['Close', 'StrategyDailyReturns', 'StrategyPortfolioValue', 'BuyHoldPortfolioValue']].tail())

Backtest simulation completed.


,Close,StrategyDailyReturns,StrategyPortfolioValue,BuyHoldPortfolioValue
Date,,,,
2019-12-11,11910.150391,NaN,100000.000000,100449.957719
2019-12-12,11971.799805,0.002588,100258.810393,100969.907580
2019-12-13,12086.700195,0.004799,100739.931723,101938.975056
2019-12-16,12053.950195,-0.001355,100603.449775,101662.762245
2019-12-17,12165.000000,0.004606,101066.866052,102599.353961


,Close,StrategyDailyReturns,StrategyPortfolioValue,BuyHoldPortfolioValue
Date,,,,
2026-04-13,23842.650391,-0.004323,150028.833172,201088.411573
2026-04-15,24231.300781,0.008150,151251.615951,204366.280787
2026-04-16,24196.750000,-0.000713,151143.783080,204074.880226
2026-04-17,24353.550781,0.003240,151633.507180,205397.334714
2026-04-20,24364.849609,0.000232,151668.682358,205492.628793


## STEP 6: Visuals

Plotting the key performance indicators and strategy components using Plotly for interactive visualization.

In [8]:
# --- Plot 1: Portfolio Growth (Strategy vs Buy & Hold) ---
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=df.index, y=df['StrategyPortfolioValue'], mode='lines', name='Strategy Portfolio Value', line=dict(color='blue')))
fig1.add_trace(go.Scatter(x=df.index, y=df['BuyHoldPortfolioValue'], mode='lines', name='Buy & Hold Portfolio Value', line=dict(color='red', dash='dash')))
fig1.update_layout(title='Portfolio Growth: Strategy vs Buy & Hold', yaxis_title='Portfolio Value (INR)', xaxis_title='Date', hovermode='x unified')
fig1.show()

# --- Plot 2: Entry Score Over Time with Threshold Lines ---
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=df.index, y=df['EntryScore'], mode='lines', name='Entry Score', line=dict(color='purple')))
fig2.add_hline(y=0.6, line_dash="dot", line_color="green", annotation_text="High Score (>0.6)", annotation_position="top right")
fig2.add_hline(y=0.3, line_dash="dot", line_color="orange", annotation_text="Medium Score (0.3-0.6)", annotation_position="bottom right")
fig2.update_layout(title='Entry Score Over Time', yaxis_title='Entry Score', xaxis_title='Date', hovermode='x unified')
fig2.show()

# --- Plot 3: Drawdown Chart for both strategies ---
def calculate_drawdown(portfolio_values):
    rolling_max = portfolio_values.cummax()
    drawdown = (portfolio_values - rolling_max) / rolling_max
    return drawdown

strategy_drawdown = calculate_drawdown(df['StrategyPortfolioValue'])
buyhold_drawdown = calculate_drawdown(df['BuyHoldPortfolioValue'])

fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=df.index, y=strategy_drawdown, mode='lines', name='Strategy Drawdown', line=dict(color='blue')))
fig3.add_trace(go.Scatter(x=df.index, y=buyhold_drawdown, mode='lines', name='Buy & Hold Drawdown', line=dict(color='red', dash='dash')))
fig3.update_layout(title='Drawdown Chart: Strategy vs Buy & Hold', yaxis_title='Drawdown (%)', xaxis_title='Date', hovermode='x unified')
fig3.show()

# --- Plot 4: NIFTY Price with Buy Zones Highlighted (green = high score) ---
fig4 = go.Figure()
fig4.add_trace(go.Scatter(x=df.index, y=df['Close'], mode='lines', name='NIFTY Close Price', line=dict(color='grey')))

# Highlight 'Buy Zones' where EntryScore > 0.6 (100% invested)
buy_zones = df[df['Position'] == 1.0]

# Use fill='toself' to highlight areas. This might be tricky with scatter if not contiguous.
# A better approach is to draw rectangles for each continuous buy zone or overlay a colored band.
# For simplicity, let's plot points for high score days.
fig4.add_trace(go.Scatter(x=buy_zones.index, y=buy_zones['Close'], mode='markers', name='High Score (100% Invested)',
                           marker=dict(color='green', size=5, symbol='circle'),
                           hoverinfo='text', hovertext='Entry Score: ' + buy_zones['EntryScore'].round(2).astype(str) + '<br>Position: 100%'))

fig4.update_layout(title='NIFTY Price with High Score Buy Zones', yaxis_title='NIFTY Close Price', xaxis_title='Date', hovermode='x unified')
fig4.show()


## STEP 7: Performance Summary

In [9]:
# --- Performance Summary ---

def calculate_cagr(start_value, end_value, num_years):
    if num_years <= 0:
        return 0
    return ((end_value / start_value) ** (1 / num_years)) - 1

def calculate_sharpe_ratio(returns, risk_free_rate=0.03):
    # Assuming daily returns, need to annualize
    annualized_return = returns.mean() * 252 # 252 trading days in a year
    annualized_std = returns.std() * np.sqrt(252)
    if annualized_std == 0:
        return 0
    return (annualized_return - risk_free_rate) / annualized_std

# Calculate performance metrics
start_date = df.index.min()
end_date = df.index.max()
num_years = (end_date - start_date).days / 365.25

# Strategy Performance
strategy_start_value = INITIAL_CAPITAL
strategy_end_value = df['StrategyPortfolioValue'].iloc[-1]
strategy_total_return = (strategy_end_value / strategy_start_value) - 1
strategy_cagr = calculate_cagr(strategy_start_value, strategy_end_value, num_years)
strategy_max_drawdown = strategy_drawdown.min()
strategy_sharpe_ratio = calculate_sharpe_ratio(df['StrategyDailyReturns'].dropna())

# Buy & Hold Performance
buyhold_start_value = INITIAL_CAPITAL
buyhold_end_value = df['BuyHoldPortfolioValue'].iloc[-1]
buyhold_total_return = (buyhold_end_value / buyhold_start_value) - 1
buyhold_cagr = calculate_cagr(buyhold_start_value, buyhold_end_value, num_years)
buyhold_max_drawdown = buyhold_drawdown.min()
buyhold_sharpe_ratio = calculate_sharpe_ratio(df['BuyHoldDailyReturns'].dropna())

# Print Summary
summary_data = {
    'Metric': ['Total Return %', 'CAGR %', 'Max Drawdown %', 'Sharpe Ratio'],
    'Strategy': [
        f"{strategy_total_return:.2%}",
        f"{strategy_cagr:.2%}",
        f"{strategy_max_drawdown:.2%}",
        f"{strategy_sharpe_ratio:.2f}"
    ],
    'Buy & Hold': [
        f"{buyhold_total_return:.2%}",
        f"{buyhold_cagr:.2%}",
        f"{buyhold_max_drawdown:.2%}",
        f"{buyhold_sharpe_ratio:.2f}"
    ]
}

summary_df = pd.DataFrame(summary_data)

print("Performance Summary:")
display(summary_df)


Performance Summary:


,Metric,Strategy,Buy & Hold
0,Total Return %,51.67%,105.49%
1,CAGR %,6.77%,12.00%
2,Max Drawdown %,-21.05%,-38.44%
3,Sharpe Ratio,0.44,0.56


In [10]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=summary_df)

https://docs.google.com/spreadsheets/d/1JQtS3Wm3XU9Y5fVNXm71DnekwOyU1Qh-yY3PE1ZpIMU/edit#gid=0


In [11]:
# Store the original DataFrame state for interactive re-runs
df_original = df.copy()

def run_interactive_analysis(momentum_weight, volatility_weight, drawdown_weight):
    global df # Declare df as global to modify it within the function

    # Reset df to its state before EntryScore calculation
    df_temp = df_original.copy()

    # Recalculate Entry Score with new weights
    df_temp = calculate_entry_score(df_temp, momentum_weight, volatility_weight, drawdown_weight)

    # Apply Position Sizing
    df_temp = apply_position_sizing(df_temp)

    # --- Backtesting ---
    INITIAL_CAPITAL = 100000
    df_temp['StrategyDailyReturns'] = df_temp['DailyReturns'] * df_temp['Position'].shift(1)
    df_temp['BuyHoldDailyReturns'] = df_temp['DailyReturns']
    df_temp['StrategyPortfolioValue'] = INITIAL_CAPITAL * (1 + df_temp['StrategyDailyReturns']).cumprod()
    df_temp['BuyHoldPortfolioValue'] = INITIAL_CAPITAL * (1 + df_temp['BuyHoldDailyReturns']).cumprod()
    df_temp['StrategyPortfolioValue'] = df_temp['StrategyPortfolioValue'].fillna(INITIAL_CAPITAL)
    df_temp['BuyHoldPortfolioValue'] = df_temp['BuyHoldPortfolioValue'].fillna(INITIAL_CAPITAL)

    # --- Calculate Drawdowns for Visuals and Summary ---
    def calculate_drawdown(portfolio_values):
        rolling_max = portfolio_values.cummax()
        drawdown = (portfolio_values - rolling_max) / rolling_max
        return drawdown

    strategy_drawdown = calculate_drawdown(df_temp['StrategyPortfolioValue'])
    buyhold_drawdown = calculate_drawdown(df_temp['BuyHoldPortfolioValue'])

    # --- Plot 1: Portfolio Growth (Strategy vs Buy & Hold) ---
    fig1 = go.Figure()
    fig1.add_trace(go.Scatter(x=df_temp.index, y=df_temp['StrategyPortfolioValue'], mode='lines', name='Strategy Portfolio Value', line=dict(color='blue')))
    fig1.add_trace(go.Scatter(x=df_temp.index, y=df_temp['BuyHoldPortfolioValue'], mode='lines', name='Buy & Hold Portfolio Value', line=dict(color='red', dash='dash')))
    fig1.update_layout(title='Portfolio Growth: Strategy vs Buy & Hold', yaxis_title='Portfolio Value (INR)', xaxis_title='Date', hovermode='x unified')
    fig1.show()

    # --- Plot 2: Entry Score Over Time with Threshold Lines ---
    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(x=df_temp.index, y=df_temp['EntryScore'], mode='lines', name='Entry Score', line=dict(color='purple')))
    fig2.add_hline(y=0.6, line_dash="dot", line_color="green", annotation_text="High Score (>0.6)", annotation_position="top right")
    fig2.add_hline(y=0.3, line_dash="dot", line_color="orange", annotation_text="Medium Score (0.3-0.6)", annotation_position="bottom right")
    fig2.update_layout(title='Entry Score Over Time', yaxis_title='Entry Score', xaxis_title='Date', hovermode='x unified')
    fig2.show()

    # --- Plot 3: Drawdown Chart for both strategies ---
    fig3 = go.Figure()
    fig3.add_trace(go.Scatter(x=df_temp.index, y=strategy_drawdown, mode='lines', name='Strategy Drawdown', line=dict(color='blue')))
    fig3.add_trace(go.Scatter(x=df_temp.index, y=buyhold_drawdown, mode='lines', name='Buy & Hold Drawdown', line=dict(color='red', dash='dash')))
    fig3.update_layout(title='Drawdown Chart: Strategy vs Buy & Hold', yaxis_title='Drawdown (%)', xaxis_title='Date', hovermode='x unified')
    fig3.show()

    # --- Plot 4: NIFTY Price with Buy Zones Highlighted (green = high score) ---
    fig4 = go.Figure()
    fig4.add_trace(go.Scatter(x=df_temp.index, y=df_temp['Close'], mode='lines', name='NIFTY Close Price', line=dict(color='grey')))
    buy_zones = df_temp[df_temp['Position'] == 1.0]
    fig4.add_trace(go.Scatter(x=buy_zones.index, y=buy_zones['Close'], mode='markers', name='High Score (100% Invested)',
                               marker=dict(color='green', size=5, symbol='circle'),
                               hoverinfo='text', hovertext='Entry Score: ' + buy_zones['EntryScore'].round(2).astype(str) + '<br>Position: 100%'))
    fig4.update_layout(title='NIFTY Price with High Score Buy Zones', yaxis_title='NIFTY Close Price', xaxis_title='Date', hovermode='x unified')
    fig4.show()

    # --- Performance Summary ---
    start_date = df_temp.index.min()
    end_date = df_temp.index.max()
    num_years = (end_date - start_date).days / 365.25

    strategy_start_value = INITIAL_CAPITAL
    strategy_end_value = df_temp['StrategyPortfolioValue'].iloc[-1]
    strategy_total_return = (strategy_end_value / strategy_start_value) - 1
    strategy_cagr = ((strategy_end_value / strategy_start_value) ** (1 / num_years)) - 1 if num_years > 0 else 0
    strategy_max_drawdown = strategy_drawdown.min()
    strategy_sharpe_ratio = (df_temp['StrategyDailyReturns'].dropna().mean() * 252 - 0.03) / (df_temp['StrategyDailyReturns'].dropna().std() * np.sqrt(252)) if (df_temp['StrategyDailyReturns'].dropna().std() * np.sqrt(252)) != 0 else 0

    buyhold_start_value = INITIAL_CAPITAL
    buyhold_end_value = df_temp['BuyHoldPortfolioValue'].iloc[-1]
    buyhold_total_return = (buyhold_end_value / buyhold_start_value) - 1
    buyhold_cagr = ((buyhold_end_value / buyhold_start_value) ** (1 / num_years)) - 1 if num_years > 0 else 0
    buyhold_max_drawdown = buyhold_drawdown.min()
    buyhold_sharpe_ratio = (df_temp['BuyHoldDailyReturns'].dropna().mean() * 252 - 0.03) / (df_temp['BuyHoldDailyReturns'].dropna().std() * np.sqrt(252)) if (df_temp['BuyHoldDailyReturns'].dropna().std() * np.sqrt(252)) != 0 else 0

    summary_data = {
        'Metric': ['Total Return %', 'CAGR %', 'Max Drawdown %', 'Sharpe Ratio'],
        'Strategy': [
            f"{strategy_total_return:.2%}",
            f"{strategy_cagr:.2%}",
            f"{strategy_max_drawdown:.2%}",
            f"{strategy_sharpe_ratio:.2f}"
        ],
        'Buy & Hold': [
            f"{buyhold_total_return:.2%}",
            f"{buyhold_cagr:.2%}",
            f"{buyhold_max_drawdown:.2%}",
            f"{buyhold_sharpe_ratio:.2f}"
        ]
    }
    summary_df = pd.DataFrame(summary_data)
    print("\nUpdated Performance Summary:")
    display(summary_df)

# Create sliders
momentum_slider = widgets.FloatSlider(min=0.0, max=1.0, step=0.05, value=0.33, description='Momentum Weight')
volatility_slider = widgets.FloatSlider(min=0.0, max=1.0, step=0.05, value=0.33, description='Volatility Weight')
drawdown_slider = widgets.FloatSlider(min=0.0, max=1.0, step=0.05, value=0.34, description='Drawdown Weight')

# Create an interactive widget layout
interactive_output = interactive(run_interactive_analysis,
                                 momentum_weight=momentum_slider,
                                 volatility_weight=volatility_slider,
                                 drawdown_weight=drawdown_slider)

# Display the sliders and output
display(VBox([HBox([momentum_slider, volatility_slider, drawdown_slider]), interactive_output]))


## BONUS: Interactive Weight Adjustment

In [12]:
# Store the original DataFrame state for interactive re-runs
df_original = df.copy()

def run_interactive_analysis(momentum_weight, volatility_weight, drawdown_weight):
    global df # Declare df as global to modify it within the function

    # Reset df to its state before EntryScore calculation
    df_temp = df_original.copy()

    # Recalculate Entry Score with new weights
    df_temp = calculate_entry_score(df_temp, momentum_weight, volatility_weight, drawdown_weight)

    # Apply Position Sizing
    df_temp = apply_position_sizing(df_temp)

    # --- Backtesting ---
    INITIAL_CAPITAL = 100000
    df_temp['StrategyDailyReturns'] = df_temp['DailyReturns'] * df_temp['Position'].shift(1)
    df_temp['BuyHoldDailyReturns'] = df_temp['DailyReturns']
    df_temp['StrategyPortfolioValue'] = INITIAL_CAPITAL * (1 + df_temp['StrategyDailyReturns']).cumprod()
    df_temp['BuyHoldPortfolioValue'] = INITIAL_CAPITAL * (1 + df_temp['BuyHoldDailyReturns']).cumprod()
    df_temp['StrategyPortfolioValue'] = df_temp['StrategyPortfolioValue'].fillna(INITIAL_CAPITAL)
    df_temp['BuyHoldPortfolioValue'] = df_temp['BuyHoldPortfolioValue'].fillna(INITIAL_CAPITAL)

    # --- Calculate Drawdowns for Visuals and Summary ---
    def calculate_drawdown(portfolio_values):
        rolling_max = portfolio_values.cummax()
        drawdown = (portfolio_values - rolling_max) / rolling_max
        return drawdown

    strategy_drawdown = calculate_drawdown(df_temp['StrategyPortfolioValue'])
    buyhold_drawdown = calculate_drawdown(df_temp['BuyHoldPortfolioValue'])

    # --- Plot 1: Portfolio Growth (Strategy vs Buy & Hold) ---
    fig1 = go.Figure()
    fig1.add_trace(go.Scatter(x=df_temp.index, y=df_temp['StrategyPortfolioValue'], mode='lines', name='Strategy Portfolio Value', line=dict(color='blue')))
    fig1.add_trace(go.Scatter(x=df_temp.index, y=df_temp['BuyHoldPortfolioValue'], mode='lines', name='Buy & Hold Portfolio Value', line=dict(color='red', dash='dash')))
    fig1.update_layout(title='Portfolio Growth: Strategy vs Buy & Hold', yaxis_title='Portfolio Value (INR)', xaxis_title='Date', hovermode='x unified')
    fig1.show()

    # --- Plot 2: Entry Score Over Time with Threshold Lines ---
    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(x=df_temp.index, y=df_temp['EntryScore'], mode='lines', name='Entry Score', line=dict(color='purple')))
    fig2.add_hline(y=0.6, line_dash="dot", line_color="green", annotation_text="High Score (>0.6)", annotation_position="top right")
    fig2.add_hline(y=0.3, line_dash="dot", line_color="orange", annotation_text="Medium Score (0.3-0.6)", annotation_position="bottom right")
    fig2.update_layout(title='Entry Score Over Time', yaxis_title='Entry Score', xaxis_title='Date', hovermode='x unified')
    fig2.show()

    # --- Plot 3: Drawdown Chart for both strategies ---
    fig3 = go.Figure()
    fig3.add_trace(go.Scatter(x=df_temp.index, y=strategy_drawdown, mode='lines', name='Strategy Drawdown', line=dict(color='blue')))
    fig3.add_trace(go.Scatter(x=df_temp.index, y=buyhold_drawdown, mode='lines', name='Buy & Hold Drawdown', line=dict(color='red', dash='dash')))
    fig3.update_layout(title='Drawdown Chart: Strategy vs Buy & Hold', yaxis_title='Drawdown (%)', xaxis_title='Date', hovermode='x unified')
    fig3.show()

    # --- Plot 4: NIFTY Price with Buy Zones Highlighted (green = high score) ---
    fig4 = go.Figure()
    fig4.add_trace(go.Scatter(x=df_temp.index, y=df_temp['Close'], mode='lines', name='NIFTY Close Price', line=dict(color='grey')))
    buy_zones = df_temp[df_temp['Position'] == 1.0]
    fig4.add_trace(go.Scatter(x=buy_zones.index, y=buy_zones['Close'], mode='markers', name='High Score (100% Invested)',
                               marker=dict(color='green', size=5, symbol='circle'),
                               hoverinfo='text', hovertext='Entry Score: ' + buy_zones['EntryScore'].round(2).astype(str) + '<br>Position: 100%'))
    fig4.update_layout(title='NIFTY Price with High Score Buy Zones', yaxis_title='NIFTY Close Price', xaxis_title='Date', hovermode='x unified')
    fig4.show()

    # --- Performance Summary ---
    start_date = df_temp.index.min()
    end_date = df_temp.index.max()
    num_years = (end_date - start_date).days / 365.25

    strategy_start_value = INITIAL_CAPITAL
    strategy_end_value = df_temp['StrategyPortfolioValue'].iloc[-1]
    strategy_total_return = (strategy_end_value / strategy_start_value) - 1
    strategy_cagr = ((strategy_end_value / strategy_start_value) ** (1 / num_years)) - 1 if num_years > 0 else 0
    strategy_max_drawdown = strategy_drawdown.min()
    strategy_sharpe_ratio = (df_temp['StrategyDailyReturns'].dropna().mean() * 252 - 0.03) / (df_temp['StrategyDailyReturns'].dropna().std() * np.sqrt(252)) if (df_temp['StrategyDailyReturns'].dropna().std() * np.sqrt(252)) != 0 else 0

    buyhold_start_value = INITIAL_CAPITAL
    buyhold_end_value = df_temp['BuyHoldPortfolioValue'].iloc[-1]
    buyhold_total_return = (buyhold_end_value / buyhold_start_value) - 1
    buyhold_cagr = ((buyhold_end_value / buyhold_start_value) ** (1 / num_years)) - 1 if num_years > 0 else 0
    buyhold_max_drawdown = buyhold_drawdown.min()
    buyhold_sharpe_ratio = (df_temp['BuyHoldDailyReturns'].dropna().mean() * 252 - 0.03) / (df_temp['BuyHoldDailyReturns'].dropna().std() * np.sqrt(252)) if (df_temp['BuyHoldDailyReturns'].dropna().std() * np.sqrt(252)) != 0 else 0

    summary_data = {
        'Metric': ['Total Return %', 'CAGR %', 'Max Drawdown %', 'Sharpe Ratio'],
        'Strategy': [
            f"{strategy_total_return:.2%}",
            f"{strategy_cagr:.2%}",
            f"{strategy_max_drawdown:.2%}",
            f"{strategy_sharpe_ratio:.2f}"
        ],
        'Buy & Hold': [
            f"{buyhold_total_return:.2%}",
            f"{buyhold_cagr:.2%}",
            f"{buyhold_max_drawdown:.2%}",
            f"{buyhold_sharpe_ratio:.2f}"
        ]
    }
    summary_df = pd.DataFrame(summary_data)
    print("\nUpdated Performance Summary:")
    display(summary_df)

# Create sliders
momentum_slider = widgets.FloatSlider(min=0.0, max=1.0, step=0.05, value=0.33, description='Momentum Weight')
volatility_slider = widgets.FloatSlider(min=0.0, max=1.0, step=0.05, value=0.33, description='Volatility Weight')
drawdown_slider = widgets.FloatSlider(min=0.0, max=1.0, step=0.05, value=0.34, description='Drawdown Weight')

# Create an interactive widget layout
interactive_output = interactive(run_interactive_analysis,
                                 momentum_weight=momentum_slider,
                                 volatility_weight=volatility_slider,
                                 drawdown_weight=drawdown_slider)

# Display the sliders and output
display(VBox([HBox([momentum_slider, volatility_slider, drawdown_slider]), interactive_output]))


## Conclusion

This project successfully developed and backtested a quantitative market timing strategy for the NIFTY 50 index. By combining momentum, volatility, and drawdown indicators into a composite 'Entry Score', the strategy aimed to identify opportune times to adjust market exposure.

The backtesting results provided a comparative analysis against a passive Buy & Hold strategy. While the strategy did not outperform the Buy & Hold in terms of total returns or CAGR over the historical period, it demonstrated a significantly lower maximum drawdown and a comparable Sharpe Ratio. This suggests that the timing strategy was effective in mitigating risk and preserving capital during market downturns, which is a crucial aspect of capital management. The interactive component further allows for exploration of how different weightings of market factors can influence the strategy's risk-reward profile.

This analysis highlights the potential benefits of a disciplined, rule-based approach to market timing, particularly for investors focused on risk management and capital preservation. Further research could involve optimizing the position sizing thresholds, exploring additional market indicators, or applying the strategy to different asset classes.